In [1]:
import traceback
import pandas as pd
import numpy as np
from sklearn.datasets import make_regression
from pycaret.regression import RegressionExperiment
from tune_grids import PYCARET_REGRESSION_TUNE_GRIDS

def shrink_grid(grid, max_choices=3):
    # take at most max_choices entries per param to keep tuning fast
    return {k: (v if not isinstance(v, (list, tuple)) else list(v)[:max_choices]) for k, v in grid.items()}

def make_sample_df(n_samples=200, n_features=6, seed=0):
    X, y = make_regression(n_samples=n_samples, n_features=n_features, noise=0.1, random_state=seed)
    cols = [f"f{i}" for i in range(n_features)]
    df = pd.DataFrame(X, columns=cols)
    df["rolling_mean_grouped_soil"] = y
    return df

def test_grids():
    df = make_sample_df(n_samples=300, n_features=6)
    rexp = RegressionExperiment()
    # very small setup -> fast checks
    s = rexp.setup(data=df,
                   target="rolling_mean_grouped_soil",
                   session_id=42,
                   train_size=0.7,
                   verbose=False,
                   n_jobs=1)

    results = {}
    for model_id, grid in PYCARET_REGRESSION_TUNE_GRIDS.items():
        print(f"\n--- Testing model_id: {model_id} ---")
        grid_small = shrink_grid(grid, max_choices=2)
        try:
            # create baseline model (may fail if model not available in this pycaret)
            base = rexp.create_model(model_id)
        except Exception as e:
            print(f" create_model failed for {model_id}: {e}")
            results[model_id] = {"status": "create_failed", "error": str(e)}
            continue

        try:
            # tune with small budget; choose_better=False to avoid extra refits
            tuned = rexp.tune_model(base, custom_grid=grid_small, choose_better=False, verbose=False)
            print(f" tuned {model_id} -> ok")
            results[model_id] = {"status": "ok", "tuned_model": str(type(tuned))}
        except Exception as e:
            print(f" tune_model failed for {model_id}: {e}")
            traceback.print_exc()
            results[model_id] = {"status": "tune_failed", "error": str(e)}

    # Summarize
    print("\nSummary:")
    for k, v in results.items():
        print(k, v["status"])
    return results

if __name__ == "__main__":
    test_grids()


--- Testing model_id: lr ---


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.0946,0.0141,0.1189,1.0000,0.0061,0.0031
1,0.0731,0.0080,0.0897,1.0000,0.0092,0.0100
2,0.0729,0.0075,0.0867,1.0000,0.0017,0.0011
3,0.1001,0.0138,0.1174,1.0000,0.0140,0.0064
4,0.0748,0.0085,0.0921,1.0000,0.0019,0.0011
5,0.0723,0.0085,0.0924,1.0000,0.0036,0.0027
6,0.0676,0.0081,0.0902,1.0000,0.0049,0.0019
7,0.0793,0.0108,0.1037,1.0000,0.0048,0.0034
8,0.0655,0.0083,0.0913,1.0000,0.0020,0.0012


 tuned lr -> ok

--- Testing model_id: lasso ---


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,1.6578,4.0727,2.0181,0.9998,0.0484,0.0262
1,2.2289,7.0031,2.6463,0.9997,0.2265,0.4946
2,1.4177,2.7876,1.6696,0.9997,0.0302,0.0231
3,2.8951,12.5713,3.5456,0.9994,0.1291,0.0677
4,2.6256,9.7129,3.1165,0.9996,0.0284,0.0237
5,1.7779,4.1884,2.0466,0.9997,0.0908,0.0775
6,2.0953,6.5141,2.5523,0.9997,0.0301,0.0245
7,2.4854,9.1748,3.0290,0.9992,0.1549,0.0802
8,1.8637,5.9786,2.4451,0.9997,0.0297,0.0236


 tuned lasso -> ok

--- Testing model_id: ridge ---


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000020 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 387
[LightGBM] [Info] Number of data points in the train set: 189, number of used features: 6
[LightGBM] [Info] Start training from score -3.270415
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,22.1247,852.9253,29.2049,0.9630,0.4146,0.3055
1,22.2277,827.8277,28.7720,0.9623,0.7237,3.6282
2,19.0919,746.5233,27.3226,0.9324,0.3699,0.2296
3,27.6983,2049.9188,45.2760,0.9055,0.5870,0.6291
4,26.8012,1290.4901,35.9234,0.9408,0.3559,0.2324
5,19.0496,605.9420,24.6159,0.9595,0.6074,0.7341
6,21.9956,812.5254,28.5048,0.9630,0.4531,0.3639
7,22.2839,1128.8635,33.5986,0.9059,0.9243,0.9104
8,25.6432,1765.3217,42.0157,0.9079,0.4395,0.2502


 tuned catboost -> ok

--- Testing model_id: dummy ---


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,119.7846,23124.2754,152.0667,-0.0030,3.1231,0.9378
1,118.9450,23263.9023,152.5251,-0.0606,3.7011,1.1879
2,97.4041,13341.5547,115.5056,-0.2076,3.2358,0.9738
3,125.2369,28124.7441,167.7043,-0.2971,2.2012,1.2301
4,129.1580,21835.8652,147.7696,-0.0015,3.6814,1.0106
5,98.7298,15958.3076,126.3262,-0.0674,2.5146,1.2412
6,126.9911,24866.9043,157.6924,-0.1333,2.4966,1.0020
7,83.6596,12342.9453,111.0988,-0.0284,3.6418,0.9857
8,103.5555,19274.0508,138.8310,-0.0059,3.5567,0.9986


 tuned dummy -> ok

Summary:
lr ok
lasso ok
ridge ok
en ok
lar ok
llar ok
omp ok
br ok
ard ok
par ok
ransac ok
tr ok
huber ok
kr ok
svm ok
knn ok
dt ok
rf ok
et ok
ada ok
gbr ok
mlp ok
xgboost ok
lightgbm ok
catboost ok
dummy ok
